[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C53_RealTime_Detectors_Course/04_rtdetr/04_rtdetr.ipynb)

# 04 · RT-DETR 解剖（NMS 耗时曲线 / hybrid encoder 计算量 / query selection / 层数可调）

目标：用**可运行的数值**把 RT-DETR 的四个设计决策各自钉死，而不是背结论。

本 notebook 你会亲手实现：
1. **numpy 版 NMS**（含确定性的 IoU 比较次数计数），实测耗时随目标数的增长曲线 → 证明 NMS 是超线性的
2. **score 阈值的精度-延迟耦合**：同一帧，阈值一变，候选数、耗时、目标覆盖率一起变
3. **端到端延迟分布**：YOLO 式（含 NMS）vs RT-DETR 式（常数），对比 p50 与 **p99**
4. **hybrid encoder 的计算量账**：attention 与 3×3 conv 的交叉点 $HW = 2.5d$，逐项验证
5. **CCFF vs 跨尺度 attention** 的反事实对比
6. **IoU-aware query selection**：可控相关性的合成实验，量化选中 query 的初始框质量
7. **decoder 层数可调**：边际收益、按预算选层数、多硬件平台的维护成本账

> 心智模型：**RT-DETR 的核心不是「更快」，是「延迟不再依赖这一帧里有什么」。**

## 1 · NMS 的代价：先把它写出来

用**IoU 比较次数**（确定性）而不是墙钟时间（有噪声）作为主要断言依据。

In [ ]:
import numpy as np, time, math
rng = np.random.default_rng(0)

def iou_1_to_n(box, boxes):
    """box: (4,) xyxy;  boxes: (N,4)  ->  (N,) IoU"""
    x1 = np.maximum(box[0], boxes[:, 0]); y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2]); y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    a0 = (box[2] - box[0]) * (box[3] - box[1])
    a1 = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return inter / np.maximum(a0 + a1 - inter, 1e-9)

def nms(boxes, scores, iou_thr=0.5):
    """标准贪心 NMS。返回 (保留下标, IoU 比较次数)。
       **比较次数是确定性的**，所以它比墙钟时间更适合做断言。"""
    order = np.argsort(-scores, kind='stable')
    keep, ops = [], 0
    while order.size > 0:
        i = order[0]; keep.append(int(i))
        rest = order[1:]
        if rest.size == 0:
            break
        ious = iou_1_to_n(boxes[i], boxes[rest])
        ops += int(rest.size)
        order = rest[ious <= iou_thr]           # 只留下与当前框不重叠的
    return np.array(keep, dtype=int), ops

# —— 手算校验（三个框，答案可以口算）——
B = np.array([[0, 0, 10, 10], [1, 1, 11, 11], [50, 50, 60, 60]], dtype=float)
S = np.array([0.9, 0.8, 0.7])
k, ops = nms(B, S, 0.5)
iou_ab = float(iou_1_to_n(B[0], B[1:2])[0])
print('IoU(A,B) = 81/119 =', round(iou_ab, 4), '  > 0.5 -> B 被抑制')
print('keep =', k.tolist(), '  IoU 比较次数 =', ops)
assert k.tolist() == [0, 2], k
assert abs(iou_ab - 81 / 119) < 1e-12
assert ops == 2
print('\n✅ NMS 就位：交 9x9=81，并 100+100-81=119')

## 2 · 耗时随目标数怎么长：合成场景 + 实测

合成规则（贴近真实检测器的输出分布）：
- 每个真实目标周围 **22 个抖动候选框**，抖动幅度与框尺寸成比例（σ = 6% 边长）
- 每个目标有一个「峰值分数」（Beta 分布），22 个候选的分数是峰值的随机折扣
- **背景误检数随场景复杂度增长**（`10 + 3·目标数`）——现实里目标多的场景杂波也多

In [ ]:
def make_scene(n_obj, rng, props_per_obj=22, bg_base=10, bg_per_obj=3, W=1920, H=1080):
    n_bg = bg_base + bg_per_obj * n_obj
    if n_obj > 0:
        ctr = rng.uniform([60, 60], [W - 60, H - 60], size=(n_obj, 2))
        sz  = rng.uniform(18, 90, size=(n_obj, 1))                  # 交通标志的典型像素尺寸
        c = np.repeat(ctr, props_per_obj, axis=0)
        s = np.repeat(sz,  props_per_obj, axis=0)
        c = c + rng.normal(0, 0.06, size=c.shape) * s               # 抖动 ∝ 框尺寸
        s = s * np.exp(rng.normal(0, 0.06, size=s.shape))
        fb = np.concatenate([c - s / 2, c + s / 2], axis=1)
        peak = rng.beta(2.2, 1.4, size=(n_obj, 1))                  # 每个目标的最高分
        fs = (peak * rng.uniform(0.25, 1.0, size=(n_obj, props_per_obj))).ravel()
        oid = np.repeat(np.arange(n_obj), props_per_obj)
    else:
        fb = np.zeros((0, 4)); fs = np.zeros(0); oid = np.zeros(0, dtype=int)
    bc = rng.uniform([0, 0], [W, H], size=(n_bg, 2))
    bs = rng.uniform(15, 60, size=(n_bg, 1))
    bb = np.concatenate([bc - bs / 2, bc + bs / 2], axis=1)
    bsc = rng.uniform(0.02, 0.35, size=n_bg)                        # 背景误检分数低
    return (np.vstack([fb, bb]), np.concatenate([fs, bsc]),
            np.concatenate([oid, np.full(n_bg, -1)]))

b0, s0, o0 = make_scene(10, np.random.default_rng(1))
print('n_obj=10 ->', len(s0), '个候选框（220 个目标候选 + 40 个背景误检）')
assert len(s0) == 10 * 22 + 40
print('✅ 场景生成器就位')

In [ ]:
N_OBJ = [2, 5, 10, 20, 40, 80, 160]
rows = []
for n in N_OBJ:
    b, s, o = make_scene(n, np.random.default_rng(100 + n))
    keep, ops = nms(b, s, 0.5)
    R = 5
    t0 = time.perf_counter()
    for _ in range(R):
        nms(b, s, 0.5)
    dt = (time.perf_counter() - t0) / R * 1000
    rows.append((n, len(s), len(keep), ops, dt))

print(f"{'目标数':>7s} {'候选框 N':>10s} {'保留 K':>8s} {'IoU 比较次数':>14s} {'numpy 耗时 ms':>15s}")
for n, N, K, ops, dt in rows:
    print(f'{n:>7d} {N:>10d} {K:>8d} {ops:>14d} {dt:>15.3f}')

n_arr   = np.array([r[0] for r in rows], dtype=float)
ops_arr = np.array([r[3] for r in rows], dtype=float)
t_arr   = np.array([r[4] for r in rows], dtype=float)
slope = float(np.polyfit(np.log(n_arr), np.log(ops_arr), 1)[0])
print(f'\n目标数 {N_OBJ[0]} -> {N_OBJ[-1]}（{N_OBJ[-1]//N_OBJ[0]} 倍），'
      f'IoU 比较次数涨了 {ops_arr[-1]/ops_arr[0]:.0f} 倍')
print(f'log-log 斜率 = {slope:.2f}    （1.0 = 线性；**> 1 就是超线性**）')
assert ops_arr[-1] > 20 * ops_arr[0]
assert slope > 1.1, slope
assert t_arr[-1] > t_arr[0]
print('\n⚠️  NMS 是 O(N·K)，而 N 和 K **都**随场景复杂度增长 -> 超线性。')
print('   TSR 场景对应：高速上 2 块标志 vs 城市路口龙门架+多块限速牌+一排店铺招牌。')

## 3 · score 阈值：精度和延迟被绑在同一个旋钮上

同一帧，只改 `score_thr`。看候选数、耗时、**目标覆盖率**（有多少真实目标还留着至少一个候选）一起变。

In [ ]:
b, s, o = make_scene(60, np.random.default_rng(7))
n_gt = len(set(o[o >= 0].tolist()))
print(f'固定场景：{n_gt} 个真实目标，{len(s)} 个原始候选\n')
print(f"{'score_thr':>10s} {'过阈 N':>9s} {'保留 K':>8s} {'IoU 比较':>10s} {'耗时 ms':>10s} {'目标覆盖率':>12s}")
covs, Ns = [], []
for thr in [0.001, 0.05, 0.15, 0.25, 0.40]:
    m = s >= thr
    bb, ss, oo = b[m], s[m], o[m]
    keep, ops = nms(bb, ss, 0.5)
    t0 = time.perf_counter()
    for _ in range(5):
        nms(bb, ss, 0.5)
    dt = (time.perf_counter() - t0) / 5 * 1000
    cov = len(set(oo[oo >= 0].tolist())) / n_gt
    covs.append(cov); Ns.append(int(m.sum()))
    print(f'{thr:>10.3f} {int(m.sum()):>9d} {len(keep):>8d} {ops:>10d} {dt:>10.3f} {cov:>11.1%}')

assert Ns == sorted(Ns, reverse=True), '阈值升高，过阈候选数必须单调下降'
assert covs[0] > covs[-1] + 0.05, '阈值升高必须付出召回代价'
print('\n⚠️  **这是一个旋钮，不是两个**：想提召回就得调低阈值 -> N 暴涨 -> NMS 变慢。')
print('   你没法「只降延迟不掉精度」。RT-DETR 的做法是把这个旋钮整个拆掉。')

## 4 · 端到端延迟分布：p50 好看，p99 才决定能不能上车

先把 NMS 的比较次数写成闭式模型（避免跑几千次真 NMS），再用它做分布模拟。

In [ ]:
def nms_ops_model(n_obj, props=22, bg_base=10, bg_per_obj=3):
    """闭式估计：目标阶段每保留 1 个框就抑制掉 props 个；背景阶段互不重叠、逐个保留。"""
    n_obj = int(n_obj)
    n_bg = bg_base + bg_per_obj * n_obj
    N = n_obj * props + n_bg
    obj_ops = sum(N - 1 - k * props for k in range(n_obj))
    bg_ops  = n_bg * (n_bg - 1) // 2
    return N, obj_ops + bg_ops

print(f"{'n_obj':>6s} {'实测比较次数':>14s} {'模型估计':>10s} {'比值':>7s}")
for n in [10, 40, 160]:
    b, s, o = make_scene(n, np.random.default_rng(200 + n))
    _, real = nms(b, s, 0.5)
    N, mdl = nms_ops_model(n)
    print(f'{n:>6d} {real:>14d} {mdl:>10d} {real/mdl:>7.2f}')
    assert 0.4 < real / mdl < 2.5, (n, real, mdl)
print('\n✅ 闭式模型与实测同量级，可以用来做上万帧的分布模拟。')

In [ ]:
# 场景目标数：重尾分布（多数帧目标少，少数城市路口帧目标极多）
rng2 = np.random.default_rng(11)
n_scene = 4000
n_obj_s = np.clip(rng2.lognormal(np.log(8), 0.9, n_scene).astype(int) + 1, 1, 220)
ops_s = np.array([nms_ops_model(int(n))[1] for n in n_obj_s], dtype=float)

nms_ms = 0.20 + 5.0e-5 * ops_s          # 标定到 GPU NMS 的量级
YOLO_FIXED = 8.30                        # 预处理 + H2D + 推理 + decode + D2H
DETR_FIXED = 11.20                       # RT-DETR：**整条链路没有依赖场景的项**
t_yolo = YOLO_FIXED + nms_ms
t_detr = np.full(n_scene, DETR_FIXED)

def stats(t):
    return dict(p50=np.percentile(t, 50), p90=np.percentile(t, 90),
                p99=np.percentile(t, 99), mean=t.mean(), mx=t.max())

sy, sd = stats(t_yolo), stats(t_detr)
print(f"{'':<16s} {'mean':>8s} {'p50':>8s} {'p90':>8s} {'p99':>8s} {'max':>8s}")
for name, st in [('YOLO 式(含NMS)', sy), ('RT-DETR 式', sd)]:
    print(f'{name:<16s} {st["mean"]:>8.2f} {st["p50"]:>8.2f} {st["p90"]:>8.2f} '
          f'{st["p99"]:>8.2f} {st["mx"]:>8.2f}')
print(f'\n目标数分布: p50={np.percentile(n_obj_s,50):.0f}  p99={np.percentile(n_obj_s,99):.0f}  '
      f'max={n_obj_s.max()}')
assert sy['p50'] < sd['p50'] - 1.0, 'YOLO 的中位延迟应明显更低'
assert sy['p99'] > sd['p99'] + 0.3,  'YOLO 的 p99 应反超 RT-DETR'
assert sy['p99'] - sy['p50'] > 2.0
assert sd['p99'] == sd['p50'], 'RT-DETR 的延迟是常数'
print('\n⚠️  **中位数上 YOLO 快 2.6 ms，p99 上反而更慢。**')
print('   按均值选型 -> 上线后在城市路口掉帧；按 p99 选型 -> 结论相反。')

In [ ]:
# 文本直方图 + 「按 p99 预留」浪费掉多少算力
lo, hi = np.percentile(t_yolo, [0.2, 99.8])
edges = np.linspace(lo, hi, 25)
h, _ = np.histogram(t_yolo, bins=edges)
print('YOLO 式端到端延迟分布（ms，条长按 count^0.4 缩放以便看清长尾）')
for i in range(len(h)):
    n = 0 if h[i] == 0 else max(1, int(60 * (h[i] / h.max()) ** 0.4))
    print(f'{edges[i]:6.2f} | {"█" * n}')
print(f'\nRT-DETR 式：全部集中在 {DETR_FIXED:.2f} ms 一根柱子上（方差为 0）')

waste = 1 - sy['p50'] / sy['p99']
print(f'\n调度器必须按 p99={sy["p99"]:.2f} ms 预留时隙，而中位帧只用掉 {sy["p50"]:.2f} ms')
print(f'-> **中位帧浪费了预留时隙的 {waste:.0%}**，而且仍然不能证明不超时（p99 不是上界）')
assert 0.15 < waste < 0.6
print('✅ 这就是「WCET 无法静态确定」在算力账上的样子。')

## 5 · Hybrid encoder 的计算量账

`attn_macs` 含 QKVO 四个投影 + $QK^\top$ + $\text{attn}\cdot V$；`conv_macs` 是同通道数的 $k\times k$ 卷积。
单位统一用 **MAC**（乘加），不是 FLOPs。

In [ ]:
D = 256
G = 1e9

def attn_macs(n_tok, d=D):
    proj = 4 * n_tok * d * d          # Q, K, V, O 四个线性投影
    sim  = n_tok * n_tok * d          # QK^T
    agg  = n_tok * n_tok * d          # attn @ V
    return proj + sim + agg

def conv_macs(n_tok, cin=D, cout=D, k=3):
    return n_tok * cin * cout * k * k

def ffn_macs(n_tok, d=D, r=4):
    return 2 * n_tok * d * (r * d)

LEVELS = [('S3 (stride 8)', 80, 80), ('S4 (stride 16)', 40, 40), ('S5 (stride 32)', 20, 20)]
print(f"{'层级':<16s} {'token':>7s} {'self-attn GMACs':>17s} {'3x3 conv GMACs':>16s} {'attn/conv':>10s}")
for name, h, w in LEVELS:
    n = h * w
    a, c = attn_macs(n), conv_macs(n)
    print(f'{name:<16s} {n:>7d} {a/G:>17.3f} {c/G:>16.3f} {a/c:>10.2f}')

n_all = sum(h * w for _, h, w in LEVELS)
assert n_all == 8400
r_all = attn_macs(n_all) / attn_macs(400)
r_sep = sum(attn_macs(h * w) for _, h, w in LEVELS) / attn_macs(400)
print(f'\n三尺度拼成一条序列({n_all} token)做全局 attention : {attn_macs(n_all)/G:8.2f} GMACs  '
      f'= AIFI 的 {r_all:.0f} 倍')
print(f'三尺度各自做 attention 再融合              : {sum(attn_macs(h*w) for _,h,w in LEVELS)/G:8.2f} GMACs  '
      f'= AIFI 的 {r_sep:.0f} 倍')
print(f'**只在 S5(400 token)做 = AIFI**            : {attn_macs(400)/G:8.3f} GMACs  = 1 倍')
assert 200 < r_all < 210, r_all
assert 128 < r_sep < 135, r_sep
print('\n✅ 205 倍。这一个数字就解释了「为什么 Deformable DETR 的 encoder 贵、RT-DETR 的不贵」。')

In [ ]:
def crossover_tokens(d=D):
    """2*(HW)^2*d + 4*HW*d^2  <  9*HW*d^2   <=>   HW < 5d/2"""
    return 5 * d // 2

hw = crossover_tokens(256)
side = int(round(hw ** 0.5))
print(f'd={D} 时的交叉点: HW = 5d/2 = {hw} 个 token  (约 {side}x{side} 的特征图)')
assert hw == 640
assert attn_macs(640) == conv_macs(640), '交叉点处两边严格相等'
assert attn_macs(639) < conv_macs(639)
assert attn_macs(641) > conv_macs(641)
print(f'  HW=639: attn {attn_macs(639):,} < conv {conv_macs(639):,}')
print(f'  HW=640: attn {attn_macs(640):,} = conv {conv_macs(640):,}   <- 严格相等')
print(f'  HW=641: attn {attn_macs(641):,} > conv {conv_macs(641):,}')

print('\n每个层级该归谁：')
for name, h, w in LEVELS:
    n = h * w
    who = '**attention 更便宜 -> AIFI**' if attn_macs(n) < conv_macs(n) else 'conv 更便宜 -> CCFF'
    print(f'  {name:<16s} {n:>5d} token   {who}')
assert attn_macs(400) < conv_macs(400)
assert attn_macs(1600) > conv_macs(1600) and attn_macs(6400) > conv_macs(6400)
print('\n✅ 「AIFI 只在 S5」不是拍脑袋，是这个不等式的直接结论。')
print('   心法：**attention 放 token 少的地方，conv 放 token 多的地方。**')

## 6 · CCFF vs 跨尺度 attention：反事实对比

In [ ]:
def fusion_block_macs(n_tok, cin=D, hidden=D // 2, n_rep=3):
    """RT-DETR 的 fusion block: 1x1 降维 -> N 个 RepBlock(3x3) -> 1x1 升维"""
    down = n_tok * cin * hidden
    rep  = n_rep * n_tok * hidden * hidden * 9
    up   = n_tok * hidden * cin
    return down + rep + up

EDGES = [('S5->S4 (自顶向下)', 1600), ('S4->S3 (自顶向下)', 6400),
         ('S3->S4 (自底向上)', 1600), ('S4->S5 (自底向上)', 400)]
tot = 0
print(f"{'融合边':<20s} {'目标层 token':>13s} {'fusion block GMACs':>20s}")
for e, n in EDGES:
    m = fusion_block_macs(n); tot += m
    print(f'{e:<20s} {n:>13d} {m/G:>20.3f}')
aifi = attn_macs(400) + ffn_macs(400)
print(f'\nCCFF 合计                      {tot/G:8.2f} GMACs')
print(f'AIFI (1 层 attn + FFN, 400 tok) {aifi/G:8.3f} GMACs')
print(f'**Hybrid encoder 合计**         {(tot+aifi)/G:8.2f} GMACs   '
      f'(其中 attention 占 {aifi/(tot+aifi):.1%})')
assert aifi / (tot + aifi) < 0.10, 'hybrid encoder 里 attention 占比应远小于 10%'

# 反事实：把 S4->S3 这条边换成 cross-attention（6400 query 对 8400 key）
edge_conv = fusion_block_macs(6400)
edge_attn = 2 * 6400 * 8400 * D
print(f'\n反事实（S4->S3 这条边）：')
print(f'  fusion block (卷积版) : {edge_conv/G:7.2f} GMACs')
print(f'  cross-attention 版    : {edge_attn/G:7.2f} GMACs   -> **贵 {edge_attn/edge_conv:.1f} 倍**')
assert edge_attn > 5 * edge_conv
print('\n✅ 「hybrid」是字面意思：encoder 里 90%+ 的计算量是卷积。')
print('   跨尺度融合本质是「把对应位置的信息对齐后相加」——卷积的归纳偏置正好合适。')

## 7 · IoU-aware query selection：起跑线有多重要

设定：encoder 输出 40000 个候选位置。每个位置有一个**真实定位质量** `iou`（与训练方式无关，固定），
和一个**分类分数** `cls`。两者的相关性 ρ 就是「分类头与定位头有多一致」：
- vanilla 训练：两个头独立优化，ρ ≈ 0.35
- IoU-aware / uncertainty-minimal 训练：显式对齐，ρ ≈ 0.90

In [ ]:
rng3 = np.random.default_rng(2024)
N_FEAT, K_QUERY = 40000, 500

z_loc = rng3.standard_normal(N_FEAT)                 # 定位质量的隐变量（固定）
iou_true = np.clip(0.34 + 0.19 * z_loc, 0.0, 1.0)    # 该位置的框与 GT 的真实 IoU

def cls_head(rho, seed):
    """分类分数：与定位质量相关性为 rho。rho 越高 = 两个头越一致。"""
    r = np.random.default_rng(seed)
    z = rho * z_loc + math.sqrt(1 - rho ** 2) * r.standard_normal(N_FEAT)
    return 1.0 / (1.0 + np.exp(-(1.1 * z - 2.2)))    # 多数位置分数很低（背景）

def selection_report(cls, iou, k=K_QUERY):
    idx = np.argsort(-cls)[:k]
    u = iou[idx]
    return dict(mean_iou=float(u.mean()),
                frac_good=float((u >= 0.5).mean()),
                n_bad=int((u < 0.2).sum()),
                hit=float(np.mean(2 * u / (1 + u))))   # 参考框内落在 GT 上的面积比例

van = selection_report(cls_head(0.35, 1), iou_true)
awa = selection_report(cls_head(0.90, 1), iou_true)
print(f"{'':<26s} {'vanilla (ρ=0.35)':>18s} {'IoU-aware (ρ=0.90)':>20s}")
print(f"{'选中 query 的平均初始 IoU':<26s} {van['mean_iou']:>18.3f} {awa['mean_iou']:>20.3f}")
print(f"{'IoU >= 0.5 的占比':<26s} {van['frac_good']:>17.1%} {awa['frac_good']:>19.1%}")
print(f"{'IoU < 0.2 的灾难性 query':<26s} {van['n_bad']:>18d} {awa['n_bad']:>20d}")
print(f"{'可变形采样点命中率 2u/(1+u)':<26s} {van['hit']:>17.1%} {awa['hit']:>19.1%}")
assert awa['mean_iou'] > van['mean_iou'] + 0.10
assert awa['frac_good'] > van['frac_good'] + 0.20
assert awa['n_bad'] < van['n_bad']
assert awa['hit'] > van['hit'] + 0.08
print('\n⚠️  vanilla 下有一批 query 从第一层开始就采不到目标 ——')
print('    可变形注意力只在参考框附近采 K 个点，参考框歪了，证据里根本没有目标。')
print('✅ 这就是「起跑线」的含义：decoder 只有 6 层，它救不回一个从零开始的定位。')

In [ ]:
# ρ 扫描：一致性 -> 起跑线质量 的完整曲线
print(f"{'ρ (两个头的一致性)':>18s} {'平均初始 IoU':>14s} {'IoU>=0.5 占比':>15s} {'采样命中率':>12s}")
prev = -1.0
for rho in [0.0, 0.2, 0.35, 0.5, 0.7, 0.9, 0.98]:
    rep = selection_report(cls_head(rho, 5), iou_true)
    print(f'{rho:>18.2f} {rep["mean_iou"]:>14.3f} {rep["frac_good"]:>14.1%} {rep["hit"]:>11.1%}')
    assert rep['mean_iou'] > prev - 0.02, '起跑线质量应随一致性单调提升'
    prev = rep['mean_iou']
print('\n✅ 论文报告这一项单独约 +0.8 AP，并且收敛更快 —— 因为 decoder 少花几层去纠错。')
print('⚠️  TSR 副作用：8x8 的框沿对角线位移 2px，IoU 就掉到 36/92 = 0.391，')
print(f'   IoU-aware 会把小目标的分数系统性压低 -> 全局 top-{K_QUERY} 里更容易落选。')
assert abs(36 / 92 - 0.3913) < 1e-3
print('   对策：按特征层级给 query 配额，或用尺度归一化的定位度量（C57 的 NWD）。')

## 8 · decoder 层数可调：一份权重，多档速度

In [ ]:
DEPTHS  = [1, 2, 3, 4, 5, 6]
DEC_AP  = [44.9, 50.5, 52.1, 52.8, 53.0, 53.1]     # 教学用示意值，量级同论文消融
DEC_LAT = [7.8, 8.1, 8.4, 8.7, 9.0, 9.3]           # ms，每层约 +0.3

print(f"{'层数':>5s} {'AP':>7s} {'延迟 ms':>9s} {'相对 6 层':>10s} {'本层新增 AP':>13s} {'AP/ms':>8s}")
gains = []
for i, d in enumerate(DEPTHS):
    dap = DEC_AP[i] - DEC_AP[i - 1] if i else float('nan')
    dms = DEC_LAT[i] - DEC_LAT[i - 1] if i else float('nan')
    per = dap / dms if i else float('nan')
    if i:
        gains.append(per)
    g1 = f'{dap:>13.1f}' if i else f"{'—':>13s}"
    g2 = f'{per:>8.1f}' if i else f"{'—':>8s}"
    print(f'{d:>5d} {DEC_AP[i]:>7.1f} {DEC_LAT[i]:>9.1f} {DEC_AP[i]-DEC_AP[-1]:>10.1f}{g1}{g2}')

assert all(gains[i] > gains[i + 1] for i in range(len(gains) - 1)), '边际收益必须递减'
print(f'\n砍掉最后 2 层：AP {DEC_AP[-1]:.1f} -> {DEC_AP[3]:.1f}（-{DEC_AP[-1]-DEC_AP[3]:.1f}），'
      f'延迟 {DEC_LAT[-1]:.1f} -> {DEC_LAT[3]:.1f} ms（省 {(1-DEC_LAT[3]/DEC_LAT[-1]):.0%}）')
print('✅ 边际收益急剧递减（18.7 -> 0.3 AP/ms），多数量产场景会毫不犹豫做这个交换。')

In [ ]:
def pick_depth(aps, lats, budget_ms):
    """给定延迟预算，返回 (层数, AP)；预算太紧则返回 (None, None)。"""
    ok = [(a, i + 1) for i, (a, l) in enumerate(zip(aps, lats)) if l <= budget_ms]
    if not ok:
        return None, None
    a, d = max(ok)
    return d, a

print('同一份权重，按平台预算裁层：')
print(f"{'平台':<20s} {'延迟预算 ms':>12s} {'选用层数':>9s} {'AP':>7s}")
for plat, bud in [('低配 SoC (共享算力)', 8.5), ('中配 SoC', 8.8),
                  ('高配 SoC', 9.5), ('极紧预算（不可行）', 7.0)]:
    d, a = pick_depth(DEC_AP, DEC_LAT, bud)
    txt = f'{d:>9d} {a:>7.1f}' if d else f"{'不可行':>9s} {'—':>7s}"
    print(f'{plat:<20s} {bud:>12.1f}{txt}')
assert pick_depth(DEC_AP, DEC_LAT, 8.5) == (3, 52.1)
assert pick_depth(DEC_AP, DEC_LAT, 9.5) == (6, 53.1)
assert pick_depth(DEC_AP, DEC_LAT, 7.0) == (None, None)
print('\n✅ 三档全部来自**同一个权重文件**，只是导出时截断到不同层数。')

In [ ]:
# 多硬件平台的工程成本账
def program_cost(n_platforms, per_train_gpu_days=8, per_eval_days=1.5, engines_per_plat=1):
    return dict(trainings=n_platforms, gpu_days=n_platforms * per_train_gpu_days,
                eval_days=n_platforms * per_eval_days,
                weights=n_platforms, engines=n_platforms * engines_per_plat)

N_PLAT = 3
yolo = program_cost(N_PLAT)                       # 每档速度训一个尺寸
rtdetr = dict(trainings=1, gpu_days=8, eval_days=N_PLAT * 1.5,
              weights=1, engines=N_PLAT)          # 一次训练；评测仍要每档跑一遍
print(f"{'':<22s} {'YOLO 系(训 3 个尺寸)':>22s} {'RT-DETR(截断 3 档)':>22s}")
for k, label in [('trainings', '训练次数'), ('gpu_days', 'GPU-天'),
                 ('eval_days', '评测人天'), ('weights', '权重文件数'), ('engines', 'engine 数')]:
    print(f'{label:<22s} {yolo[k]:>22} {rtdetr[k]:>22}')
assert rtdetr['gpu_days'] * 3 == yolo['gpu_days']
assert rtdetr['weights'] == 1 and yolo['weights'] == N_PLAT
print('\n✅ 一份权重 = 一条数据管线、一次训练预算、一套 badcase 归因、一份安全评审证据链。')
print('   注意：**评测人天并没有省** —— 每一档都必须独立评测，这一点常被忽略。')

## ✏️ 练习 1：NMS 代价的闭式模型

实现 `nms_cost(n_obj, props=22, bg_base=10, bg_per_obj=3)` 返回 `(N, ops)`：
- `n_bg = bg_base + bg_per_obj * n_obj`；`N = n_obj*props + n_bg`
- 目标阶段：第 k 轮（k 从 0 起）比较 `N - 1 - k*props` 次，共 `n_obj` 轮
- 背景阶段：`n_bg` 个互不重叠的框，比较次数 `n_bg*(n_bg-1)//2`

In [ ]:
def nms_cost(n_obj, props=22, bg_base=10, bg_per_obj=3):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert nms_cost(0) == (10, 45), nms_cost(0)                    # 10*9//2 = 45
assert nms_cost(1) == (35, 34 + 78), nms_cost(1)               # 22+13=35; 34; 13*12//2=78
assert nms_cost(8) == (210, 1056 + 561), nms_cost(8)           # 8*209-22*28=1056; 34*33//2=561
_, o10 = nms_cost(10)
_, o80 = nms_cost(80)
assert o10 == 1600 + 780, o10
assert o80 / o10 > 8, '目标数 x8，比较次数应涨远超 8 倍（超线性）'
print(f'n_obj=10 -> ops={o10};  n_obj=80 -> ops={o80};  倍数 {o80/o10:.1f}（目标数只涨了 8 倍）')
print('✅ 练习 1 通过：NMS 是 O(N·K)，而 N 与 K 都随场景增长。')

## ✏️ 练习 2：attention / conv 的归属判定

实现三个函数：
- `crossover_tokens(d)` → `5*d//2`
- `attention_cheaper(n_tok, d)` → `attn_macs(n_tok,d) < conv_macs(n_tok,...)`（**严格小于**）
- `plan_encoder(levels, d)` → `{层级名: 'AIFI' 或 'CCFF'}`，`levels` 是 `{名字: token 数}`

In [ ]:
def crossover_tokens(d=256):
    # TODO
    raise NotImplementedError

def attention_cheaper(n_tok, d=256):
    # TODO
    raise NotImplementedError

def plan_encoder(levels, d=256):
    # TODO: attention 更便宜的层 -> 'AIFI'，否则 -> 'CCFF'
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert crossover_tokens(256) == 640
assert crossover_tokens(128) == 320
assert attention_cheaper(400) and attention_cheaper(639)
assert not attention_cheaper(640), '交叉点处两边相等，严格小于应为 False'
assert not attention_cheaper(1600) and not attention_cheaper(6400)
plan = plan_encoder({'S3': 6400, 'S4': 1600, 'S5': 400})
assert plan == {'S3': 'CCFF', 'S4': 'CCFF', 'S5': 'AIFI'}, plan
# d 变大 -> 交叉点右移 -> 更多层适合 attention
plan_big = plan_encoder({'S3': 6400, 'S4': 1600, 'S5': 400}, d=1024)
assert plan_big['S4'] == 'AIFI', 'd=1024 时交叉点=2560 token，S4(1600) 也该归 attention'
print('d=256  ->', plan)
print('d=1024 ->', plan_big, '  （交叉点 =', crossover_tokens(1024), 'token）')
print('✅ 练习 2 通过：AIFI 的层级归属是一个不等式，不是设计者的偏好。')

## ✏️ 练习 3：query selection 的质量评估

实现 `selection_quality(cls, iou, k)` 返回 `(mean_iou, frac_good, n_bad)`：
按 `cls` 降序取 top-k，统计这 k 个位置的平均 `iou`、`iou>=0.5` 的比例、`iou<0.2` 的个数。

In [ ]:
def selection_quality(cls, iou, k):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测（手算）——
c = np.array([0.90, 0.20, 0.80, 0.10, 0.55])
u = np.array([0.30, 0.90, 0.70, 0.95, 0.15])
m, g, nb = selection_quality(c, u, 3)          # top3 by cls -> idx 0,2,4 -> iou 0.30,0.70,0.15
assert abs(m - (0.30 + 0.70 + 0.15) / 3) < 1e-12, m
assert abs(g - 1 / 3) < 1e-12, g               # 只有 0.70 >= 0.5
assert nb == 1, nb                             # 只有 0.15 < 0.2
m5, g5, nb5 = selection_quality(c, u, 5)
assert abs(m5 - u.mean()) < 1e-12
assert abs(g5 - 0.6) < 1e-12 and nb5 == 1
# 用第 7 节的合成数据复现主结论
mv, gv, bv = selection_quality(cls_head(0.35, 3), iou_true, 500)
ma, ga, ba = selection_quality(cls_head(0.90, 3), iou_true, 500)
print(f'vanilla   mean_iou={mv:.3f}  good={gv:.1%}  bad={bv}')
print(f'IoU-aware mean_iou={ma:.3f}  good={ga:.1%}  bad={ba}')
assert ma > mv + 0.10 and ga > gv + 0.20 and ba < bv
print('✅ 练习 3 通过：起跑线质量可以直接量化，不用等 AP 出来。')

## ✏️ 练习 4：速度菜单与预算选层

实现 `speed_menu(aps, lats, budgets)`：对每个预算返回 `(budget, depth, ap)`，
不可行时 `(budget, None, None)`。规则同 `pick_depth`：在满足 `lat <= budget` 的层数里取 AP 最大的。

In [ ]:
def speed_menu(aps, lats, budgets):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
menu = speed_menu(DEC_AP, DEC_LAT, [7.0, 8.0, 8.5, 9.0, 12.0])
expect = [(7.0, None, None), (8.0, 1, 44.9), (8.5, 3, 52.1), (9.0, 5, 53.0), (12.0, 6, 53.1)]
assert menu == expect, menu
for b, d, a in menu:
    line = f'{d} 层 -> AP {a}' if d else '不可行（连 1 层都超预算）'
    print(f'预算 {b:>5.1f} ms : {line}')
# 一份权重覆盖了几档？
covered = len({d for _, d, _ in menu if d is not None})
assert covered == 4
print(f'\n✅ 练习 4 通过：同一份权重覆盖了 {covered} 档不同的速度-精度工作点。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def nms_cost(n_obj, props=22, bg_base=10, bg_per_obj=3):
    n_obj = int(n_obj)
    n_bg = bg_base + bg_per_obj * n_obj
    N = n_obj * props + n_bg
    obj_ops = sum(N - 1 - k * props for k in range(n_obj))
    bg_ops = n_bg * (n_bg - 1) // 2
    return N, obj_ops + bg_ops

In [ ]:
# 练习 2 参考答案
def crossover_tokens(d=256):
    return 5 * d // 2

def attention_cheaper(n_tok, d=256):
    return attn_macs(n_tok, d) < conv_macs(n_tok, cin=d, cout=d, k=3)

def plan_encoder(levels, d=256):
    return {name: ('AIFI' if attention_cheaper(n, d) else 'CCFF')
            for name, n in levels.items()}

In [ ]:
# 练习 3 参考答案
def selection_quality(cls, iou, k):
    idx = np.argsort(-np.asarray(cls))[:k]
    u = np.asarray(iou)[idx]
    return float(u.mean()), float((u >= 0.5).mean()), int((u < 0.2).sum())

In [ ]:
# 练习 4 参考答案
def speed_menu(aps, lats, budgets):
    out = []
    for b in budgets:
        ok = [(a, i + 1) for i, (a, l) in enumerate(zip(aps, lats)) if l <= b]
        if ok:
            a, d = max(ok)
            out.append((b, d, a))
        else:
            out.append((b, None, None))
    return out

---
## 🧪 真实工程胶囊：RT-DETR 上车前的检查单与配置片段

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# RT-DETR 上车检查单（按顺序做，每一步都有明确的通过条件）
# ══════════════════════════════════════════════════════════════════════

# ① 导出：确认 decoder 层数可裁，并为每一档单独导出
#    mmdet/ultralytics/lyuwenyu 各家实现的字段名不同，但都是同一个开关
#    (lyuwenyu/RT-DETR 的 yaml，字段名以你用的实现为准)
#
#    RTDETRTransformer:
#      num_decoder_layers: 6      # 训练用 6
#      eval_idx: -1               # ← **推理时取第几层的头**：-1=最后一层, 2=第3层
#      num_queries: 300           # 上限：单帧最多能检出 300 个目标
#      feat_strides: [8, 16, 32]  # S3/S4/S5
#    HybridEncoder:
#      use_encoder_idx: [2]       # ← **AIFI 只作用在索引 2（=S5）**，别改成 [0,1,2]
#      num_encoder_layers: 1
#      expansion: 1.0             # CCFF fusion block 的宽度
#
# 通过条件：改 eval_idx 后重新导出，ONNX 的节点数应显著减少；AP 掉幅符合预期消融表

# ② 算子检查（DETR 系最容易在这里卡住）
#    grid_sample / MultiScaleDeformableAttention 在部分后端需要 plugin
#    RT-DETRv2 的离散采样版本可以绕开，但要确认你用的是哪个分支
#   $ polygraphy inspect model rtdetr.onnx --show layers | grep -i -E "grid|deform"
#   $ trtexec --onnx=rtdetr.onnx --fp16 --verbose 2>&1 | grep -i -E "unsupported|plugin|subgraph"
# 通过条件：子图数 == 1（没有静默回退），无 unsupported layer

# ③ **延迟稳定性验证**（这是选 RT-DETR 的理由，必须验证它真的成立）
#    用一组目标数差异极大的真实帧跑，而不是同一张图跑 1000 次
#   for img in [highway_2signs.jpg, gantry_8signs.jpg, urban_40boxes.jpg]:
#       latencies = [timed_infer(img) for _ in range(500)]   # 含 warmup 丢弃
#       p50, p99 = np.percentile(latencies, [50, 99])
# 通过条件：**不同帧之间的 p50 差异 < 3%**（RT-DETR 应该做到）
#           如果差异大，说明你的预处理或后处理里混进了数据依赖项

# ④ 与 YOLO baseline 的公平对比（同口径！见模块 05）
#    必须统一：batch=1 / 含预处理与拷贝 / 含 NMS / 同精度 / 同卡 / 同 warmup
#    报 **p50 和 p99 两个数**，只报一个的对比无效

# ⑤ TSR 特有检查
#    · num_queries=300 是硬天花板 -> 统计你的数据集里单帧最大标志数，留 3x 余量
#    · 按**像素尺寸分桶**评测（<16px / 16-32 / 32-64 / >64），COCO 口径的 AP 会掩盖小目标退化
#    · IoU-aware 会压低小目标分数 -> 检查 top-300 里小目标 query 的占比是否够
#      selected_small_ratio = (选中 query 的 GT 面积 < 32^2 的比例) / (数据集里小目标占比)
#      这个比值明显 < 1 就说明小目标在 query selection 阶段就被挤掉了

# ⑥ 多档速度的发布纪律
#    每一档 (eval_idx) 都是一个**独立的发布物**，必须各自：
#    过完整评测集 / 过分场景切片 / 过回归门禁 / 记录到模型卡
#    「一份权重」省的是训练，不是评测。
'''
print(RECIPE)
for token in ['eval_idx', 'use_encoder_idx', 'num_queries', 'grid_sample',
              'p50 和 p99', '分桶', 'subgraph']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：层数可裁 / AIFI 层级 / 算子回退 / 延迟稳定性 / 公平对比 / TSR 专项 / 发布纪律')

### 小结

- **RT-DETR 的卖点不是「平均更快」，是「延迟不再依赖这一帧里有什么」。**
  NMS 是 $O(NK)$，实测 log-log 斜率约 1.7；目标数翻 10 倍，比较次数涨约 50 倍。
  于是 **p50 上 YOLO 更快，p99 上反而更慢**——按均值选型会上线才发现掉帧。
- **AIFI 只在 S5**：三尺度拼成 8400 token 做全局 attention，代价是只在 400 token 上做的 **205 倍**。
  更精确的判据是 $2(HW)^2d + 4HWd^2 < 9HWd^2 \iff HW < \tfrac52 d$，$d=256$ 时交叉点 **640 token**，
  S5 的 400 在下面、S4 的 1600 在上面。**心法：attention 放 token 少的地方，conv 放 token 多的地方。**
- **CCFF 全用卷积**：跨尺度融合是「对齐后相加」的局部操作，卷积的归纳偏置正合适；
  换成 cross-attention 单是 S4→S3 一条边就贵 8 倍。hybrid encoder 里 attention 占比 < 10%。
- **query selection 是 decoder 的起跑线**：可变形注意力只在参考框附近采样，
  参考框 IoU 0.25 时只有 40% 的采样面积落在目标上。把两个头的一致性从 0.35 提到 0.90，
  选中 query 的平均初始 IoU 从 0.51 涨到 0.79。**副作用：小目标分数被系统性压低。**
- **逐层辅助头 = 一份权重多档速度**：边际收益急剧递减（18.7 → 0.3 AP/ms），
  砍最后 2 层掉 0.3 AP、省 6% 延迟。**省的是训练与权重管理，不省评测**。
- **选型没有普适答案**：硬约束是「p99 可静态界定」就选 RT-DETR 系；
  硬约束是「极小算力 + 极小目标 + 零 plugin」就选 RTMDet/YOLO。
  TSR 量产系统常见的第三条路是两级架构——把延迟不确定性挪到一个能设硬上限的地方。

下一站：**模块 05 · 延迟-精度权衡与实时检测器选型** —— 为什么论文的 FPS 不可信，以及怎么自己测。